# M4S3 - HPC and Machine Learning
## Training is not only an ML problem - it is a systems problem

**Predict → run → observe → discuss**

CPU-safe core examples:
1. environment and accelerator check;
2. tiny ML training loop;
3. Python loops vs optimized NumPy;
4. preprocessing bottleneck;
5. batch-size trade-off;
6. prefetch/overlap;
7. precision and memory;
8. model training-state memory budget;
9. checkpointing;
10. real PyTorch through the cluster module system;
11. preview of data-parallel gradient averaging;
12. optional Slurm/GPU resource inspection.

> Required route: NumPy only. Cluster PyTorch and GPU-related cells are optional and fail safely if the current Jupyter session has no GPU allocation.


## 0 - Check the environment
**Predict:** CPU, memory, storage or GPU - what will limit this run?


In [ ]:
import os,sys,shutil,subprocess,time,importlib.util
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np
from IPython.display import display, HTML
def sh(cmd):
    p=subprocess.run(cmd,shell=True,text=True,capture_output=True); return (p.stdout+p.stderr).strip()
print("host:",sh("hostname")); print("Python:",sys.version.split()[0]); print("NumPy:",np.__version__)
print("CPU cores:",os.cpu_count()); print("Slurm:",bool(shutil.which("sbatch"))); print("nvidia-smi:",shutil.which("nvidia-smi"))
if shutil.which("nvidia-smi"): print(sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader | head -4"))
cmd='bash -lc \'if type module >/dev/null 2>&1; then module -t avail 2>&1 | grep -Ei "pytorch|tensorflow|cuda|cudnn|nccl" | head -20; else echo "module command not visible"; fi\''
print("\nRelevant modules:\n"+sh(cmd))

def show_bars(title, items, unit="", digits=1):
    vals=[float(v) for _,v in items]
    m=max(vals) if vals else 1.0
    html=f"<div style='font-family:Arial,sans-serif;max-width:820px'><h4>{title}</h4>"
    for label,val in items:
        width=0 if m==0 else 100*float(val)/m
        value=f"{float(val):,.{digits}f}{unit}"
        html += (
            "<div style='display:grid;grid-template-columns:150px 1fr 170px;gap:10px;"
            "align-items:center;margin:5px 0'>"
            f"<div>{label}</div>"
            "<div style='background:#e8edf3;height:18px;border-radius:2px'>"
            f"<div style='width:{width:.1f}%;background:#55758f;height:18px;border-radius:2px'></div></div>"
            f"<div style='text-align:right'>{value}</div></div>"
        )
    display(HTML(html+"</div>"))


## 1 - A tiny ML workload
We use a small softmax classifier. The model is intentionally simple: today we care about **data movement, batching, memory and throughput**, not network architecture.


In [ ]:
rng=np.random.default_rng(42); N,D,C=12000,96,5
X=rng.normal(size=(N,D)).astype(np.float32); wt=rng.normal(size=(D,C)).astype(np.float32)
y=np.argmax(X@wt+0.7*rng.normal(size=(N,C)),axis=1).astype(np.int64)
cut=9600; Xtr,Xv=X[:cut],X[cut:]; ytr,yv=y[:cut],y[cut:]
def softmax(z):
    z=z-z.max(1,keepdims=True); e=np.exp(z); return e/e.sum(1,keepdims=True)
def acc(X,y,W,b): return float(np.mean(np.argmax(X@W+b,1)==y))
def grad(X,y,W,b):
    p=softmax(X@W+b); n=len(X); loss=-np.log(p[np.arange(n),y]+1e-12).mean()
    p[np.arange(n),y]-=1; p/=n; return float(loss),X.T@p,p.sum(0)
def train(bs=128,epochs=4,lr=.18,seed=0):
    rg=np.random.default_rng(seed); W=np.zeros((D,C),np.float32); b=np.zeros(C,np.float32); updates=0; t=time.perf_counter()
    for _ in range(epochs):
        ids=rg.permutation(len(Xtr))
        for s in range(0,len(Xtr),bs):
            q=ids[s:s+bs]; _,gW,gb=grad(Xtr[q],ytr[q],W,b); W-=lr*gW; b-=lr*gb; updates+=1
    return W,b,time.perf_counter()-t,updates
W,b,t,u=train()
print(f"samples={len(Xtr):,}, features={D}, data={Xtr.nbytes/2**20:.2f} MiB")
print(f"time={t:.3f}s, updates={u}, throughput={len(Xtr)*4/t:,.0f} samples/s, val_acc={acc(Xv,yv,W,b):.3f}")


## 2 - [DEMO] Optimized libraries
**Predict:** how much faster is NumPy matrix multiplication than nested Python loops?


In [ ]:
A=rng.normal(size=(1200,256)); v=rng.normal(size=256)
def py_mv(A,v):
    out=np.empty(len(A))
    for i,row in enumerate(A):
        s=0.
        for j,x in enumerate(row): s+=x*v[j]
        out[i]=s
    return out
t=time.perf_counter(); a=py_mv(A,v); tp=time.perf_counter()-t
t=time.perf_counter(); 
for _ in range(30): b=A@v
tn=(time.perf_counter()-t)/30
print(f"Python loops: {tp:.4f}s | NumPy: {tn:.6f}s | speedup: {tp/tn:,.0f}x | same result: {np.allclose(a,b)}")


**Lesson:** PyTorch/TensorFlow Python code often calls highly optimized compiled kernels underneath.


## 3 - [DEMO] Preprocessing can starve the accelerator


In [ ]:
raw=rng.integers(0,256,size=(900,96),dtype=np.uint8)
def slow(a):
    z=np.empty(a.shape,np.float32)
    for i in range(len(a)):
        for j in range(a.shape[1]): z[i,j]=(float(a[i,j])-127.5)/127.5
    return z
def fast(a): return (a.astype(np.float32)-127.5)/127.5
t=time.perf_counter(); z1=slow(raw); ts=time.perf_counter()-t
t=time.perf_counter(); z2=fast(raw); tf=time.perf_counter()-t
print(f"slow={len(raw)/ts:,.0f} samples/s | vectorized={len(raw)/tf:,.0f} samples/s | speedup={ts/tf:.1f}x")
for prep in (30,12,4):
    compute=12; print(f"prep={prep:2} ms, compute=12 ms -> approx accelerator utilization={100*compute/(prep+compute):.1f}%")


**Fix the feeding pipeline before buying more GPUs.**


## 4 - [DEMO] Batch size: throughput ↔ memory ↔ convergence
Train the same model for the same 4 epochs.


In [ ]:
rows=[]
for bs in (16,64,256,1024):
    W,b,t,u=train(bs=bs,epochs=4,seed=1)
    rows.append((bs,u,t,len(Xtr)*4/t,acc(Xv,yv,W,b),bs*D*4/1024))
print(f"{'batch':>6} {'updates':>8} {'time':>8} {'samples/s':>12} {'val_acc':>8} {'batch_KiB':>10}")
for r in rows: print(f"{r[0]:6d} {r[1]:8d} {r[2]:8.3f} {r[3]:12,.0f} {r[4]:8.3f} {r[5]:10.1f}")


In [ ]:
show_bars(
    "Measured throughput by batch size",
    [(f"batch {r[0]}", r[3]) for r in rows],
    unit=" samples/s",
    digits=0
)
show_bars(
    "Approximate input-batch memory",
    [(f"batch {r[0]}", r[5]) for r in rows],
    unit=" KiB",
    digits=1
)
print("Validation accuracy:", {r[0]: round(r[4],3) for r in rows})
print("Question: which batch is best if throughput and validation accuracy both matter?")


Discuss:
- larger batches usually improve hardware utilization;
- memory use grows;
- updates per epoch fall;
- convergence can change;
- the batch with the highest throughput is not necessarily the batch with the best validation result.

**Batch size is both an ML parameter and a systems parameter.**


## 5 - [DEMO] Prefetch hides I/O behind compute


In [ ]:
def load(i): time.sleep(.018); return i
def compute(x): time.sleep(.030)
n=12; t=time.perf_counter()
for i in range(n): compute(load(i))
seq=time.perf_counter()-t
t=time.perf_counter()
with ThreadPoolExecutor(max_workers=1) as ex:
    f=ex.submit(load,0)
    for i in range(n):
        x=f.result()
        if i+1<n: f=ex.submit(load,i+1)
        compute(x)
pre=time.perf_counter()-t
print(f"sequential={seq:.3f}s | prefetched={pre:.3f}s | speedup={seq/pre:.2f}x")


Prefetching does not make I/O or compute individually faster; it improves **pipeline utilization** by overlapping them.


## 6 - [DEMO] Precision changes memory and numerical behaviour


In [ ]:
base=rng.normal(size=(2048,512))
for name,dtype in (("FP64",np.float64),("FP32",np.float32),("FP16",np.float16)):
    a=base.astype(dtype); print(f"{name}: {a.nbytes/2**20:.2f} MiB")
A=base[:512,:256]; B=base[:256,:256]; ref=A@B
for name,dtype in (("FP32",np.float32),("FP16",np.float16)):
    a=A.astype(dtype); b=B.astype(dtype); t=time.perf_counter(); out=a@b; dt=time.perf_counter()-t
    err=np.linalg.norm(out.astype(np.float64)-ref)/np.linalg.norm(ref)
    print(f"{name}: time={dt:.4f}s, relative_difference={err:.3e}")


In [ ]:
precision_memory=[("FP64",8.0),("FP32",4.0),("FP16",2.0)]
show_bars("Memory footprint for the same array", precision_memory, unit=" MiB", digits=1)
print("Important: lower precision reduces memory, but speed depends on hardware support.")


> On CPUs FP16 may not be faster. GPUs can have specialized tensor hardware, so **hardware support matters**.


## 7 - [DEMO] How much memory does training state need?

A model needs more memory for training than for storing its parameters alone.

For a simplified FP32 model trained with Adam:

- parameters: **4 bytes / parameter**
- gradients: **4 bytes / parameter**
- Adam first moment: **4 bytes / parameter**
- Adam second moment: **4 bytes / parameter**

Simplified training-state estimate:

**about 16 bytes per parameter**

This excludes activations, temporary buffers, framework overhead and other copies.

**Predict:** could a 7-billion-parameter model train in this simplified form on one 24 GB GPU?


In [ ]:
models=[
    ("1 million",1_000_000),
    ("100 million",100_000_000),
    ("1 billion",1_000_000_000),
    ("7 billion",7_000_000_000),
]
bytes_per_parameter=16
print(f"{'model size':>14} {'training state':>18}")
memory_rows=[]
for label,nparams in models:
    gb=nparams*bytes_per_parameter/1e9
    memory_rows.append((label,gb))
    print(f"{label:>14} {gb:15.3f} GB")
show_bars("Simplified FP32 + Adam training-state memory", memory_rows, unit=" GB", digits=1)
print("\nA 7B model needs about 112 GB for this simplified training state alone.")
print("Activations and temporary tensors would require additional memory.")


**Key idea:** model size can become an HPC memory problem before compute becomes the main problem.

Mixed precision, sharding and distributed strategies can reduce or distribute parts of this footprint. The distributed strategies belong to the next session.


## 8 - [DEMO] Checkpointing is resilience + I/O


In [ ]:
out=Path("m4s3_outputs"); out.mkdir(exist_ok=True); f=out/"checkpoint.npz"
W,b,_,_=train(bs=128,epochs=2,seed=3); before=acc(Xv,yv,W,b)
t=time.perf_counter(); np.savez(f,W=W,b=b,epoch=np.array([2])); save=time.perf_counter()-t
del W,b; t=time.perf_counter(); s=np.load(f); load=time.perf_counter()-t
after=acc(Xv,yv,s["W"],s["b"])
print(f"file={f} | size={f.stat().st_size/1024:.1f} KiB | save={save*1e3:.2f} ms | load={load*1e3:.2f} ms")
print(f"accuracy before={before:.4f}, after reload={after:.4f}, recovered={np.isclose(before,after)}")


At scale, checkpoint frequency trades **recovery time** against **storage/I/O overhead**.

Real training jobs may run for hours or days, so losing all progress after a failure can be very expensive.


## 9 - [REAL CLUSTER] PyTorch from the module system

The environment check can show PyTorch modules even when the current Jupyter kernel cannot import PyTorch directly.

This demonstrates two different HPC concepts:

**software availability ≠ software loaded into this process**

**GPU exists in the cluster ≠ GPU allocated to this job**

The cell below launches a separate shell, loads a cluster PyTorch module, then runs a real PyTorch matrix multiplication.

**Predict:** if this Jupyter session has no GPU allocation, what device will PyTorch use?


In [ ]:
cluster_script=r'''
set +e
module load pytorch/2.8.0 >/dev/null 2>&1
if [ $? -ne 0 ]; then
  module load PyTorch/2.1.2-foss-2023a >/dev/null 2>&1
fi
python - <<'PY'
import time
try:
    import torch
except Exception as e:
    print("PyTorch import failed after module load:", repr(e))
    raise SystemExit(0)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
device="cuda" if torch.cuda.is_available() else "cpu"
if device=="cuda":
    print("GPU:", torch.cuda.get_device_name(0))
n=1800 if device=="cuda" else 900
a=torch.randn((n,n),device=device)
b=torch.randn((n,n),device=device)
if device=="cuda":
    torch.cuda.synchronize()
t=time.perf_counter()
c=a@b
if device=="cuda":
    torch.cuda.synchronize()
print(f"device={device} | matmul={n}x{n} | time={time.perf_counter()-t:.4f}s")
PY
'''
p=subprocess.run(["bash","-lc",cluster_script],text=True,capture_output=True)
print((p.stdout+p.stderr).strip())
if p.returncode!=0:
    print("Cluster module demo returned code",p.returncode)


If CUDA reports **False**, that does not prove the cluster has no GPUs.

It means this process currently has no CUDA device available. In HPC, software modules and hardware allocations are separate parts of the workflow.


## 10 - [PREVIEW] Data-parallel gradient averaging
Session 19 will study All-Reduce and distributed training in detail.


In [ ]:
ids=rng.choice(len(Xtr),512,replace=False); xb,yb=Xtr[ids],ytr[ids]
W=np.zeros((D,C),np.float32); b=np.zeros(C,np.float32)
_,gf,bf=grad(xb,yb,W,b); gs=[]; bs=[]
for q in np.array_split(np.arange(512),4):
    _,g,bb=grad(xb[q],yb[q],W,b); gs.append(g); bs.append(bb)
ga=np.mean(gs,0); ba=np.mean(bs,0)
print("4 workers x 128 samples")
print("max gradient difference:",float(np.max(np.abs(gf-ga))))
print("max bias-gradient difference:",float(np.max(np.abs(bf-ba))))


**That averaging step is where collective communication such as All-Reduce enters.**

The numerical differences above should be extremely small because averaging the gradients from equal-sized shards reproduces the full-batch gradient up to floating-point round-off.

Session 19 will study how this synchronization is implemented and how communication limits scaling.


## 11 - [OPTIONAL CLUSTER] What would an ML job request from Slurm?

A GPU training job usually needs more than a GPU.

Typical resources include:

- **GPU** for model computation;
- **CPU cores** for loading and preprocessing;
- **memory** for batches and worker processes;
- **storage bandwidth** for datasets and checkpoints.

A generic request might look like:

```bash
#SBATCH --job-name=ml-demo
#SBATCH --gpus=1
#SBATCH --cpus-per-task=8
#SBATCH --mem=32G
```

The exact GPU partition and options are cluster-specific, so inspect the system before submitting.


In [ ]:
if shutil.which("sinfo"):
    print("Visible Slurm partitions and generic resources:")
    print(sh("sinfo -o '%P  GRES=%G  CPUs=%c  MemMB=%m' | head -15"))
else:
    print("sinfo is not visible in this environment.")

print("\nQuestion: why request CPU cores for a GPU training job?")
print("Expected: loading, decoding, preprocessing, augmentation and orchestration can run on CPUs.")


# 12 - Exit ticket

A training job shows:
- GPU utilization **35%**
- GPU memory **45%**
- CPU utilization **100%**
- continuous storage reads

**Would adding more GPUs be your first change? Why or why not?**

Expected: **probably not**. The job appears CPU/data-feed limited, so additional GPUs may simply wait.

### Remember

**pipeline → bottleneck → evidence → change one factor → remeasure**

### Bridge to M4S4

One GPU can run out of compute capacity or memory. Multiple GPUs can help, but they introduce **communication and synchronization**.

Next: **distributed ML and AI workflows**
